Link: https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### 🕰️ Legacy Chains

> **LangChain 1.x**: `LLMChain` is a *retired* API. It no longer ships in the
> `langchain` package — it lives in the compatibility package
> `langchain-classic`, which is why the import below reads
> `from langchain_classic.chains.llm import LLMChain`.
>
> It still runs, and it is shown here on purpose so you can compare it against
> the LCEL version in the next section. **Do not reach for it in new code.**

In [ ]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")],
)

legacy_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini"), prompt=prompt)

legacy_result = legacy_chain({"adjective": "funny"})
legacy_result

### ✨ LCEL — the 1.x way

The same behaviour, composed with the pipe operator instead of a chain class.
LCEL is **not** deprecated: `prompt | llm | parser` is the current, supported
form, and it streams and batches without extra work.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")],
)

chain = prompt | ChatOpenAI(model="gpt-4o-mini")| StrOutputParser()

chain.invoke({"adjective": "funny"})

### 🕰️ Legacy RAG

> **LangChain 1.x**: `RetrievalQA` is retired the same way `LLMChain` is, and
> imports from `langchain-classic`. The prompt hub (`from langchain_classic
> import hub`) moved with it.
>
> Kept here as the "before" half of the comparison — the LCEL rewrite follows.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma


embedding_function = OpenAIEmbeddings()

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embedding_function)
llm = ChatOpenAI(model="gpt-4o-mini")
retriever = db.as_retriever()

In [ ]:
from langchain_classic import hub
from langchain_classic.chains import RetrievalQA

prompt = hub.pull("rlm/rag-prompt")

qa_chain = RetrievalQA.from_llm(llm, retriever=retriever, prompt=prompt) # RetrievalQA is already deprecated

qa_chain("What does the cat like to eat?")

### ✨ LCEL — the 1.x way

The retrieval equivalent, composed rather than packaged. Note that the retriever
is just another runnable in the pipe.

> **Why does this "modern" cell still import from `langchain-classic`?**
> Only for `hub`. The *chain* is fully LCEL — but the **prompt hub** moved to
> `langchain-classic` along with the rest of the retired surface, so pulling a
> community prompt (`hub.pull("rlm/rag-prompt")`) still goes through that package.
> To drop the dependency entirely, inline the prompt with `ChatPromptTemplate`
> instead of pulling it; that also makes the notebook run without network access.

In [ ]:
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = hub.pull("rlm/rag-prompt")


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

qa_chain.invoke("What does the cat like to eat?")